In [5]:
1+1

2

In [6]:
using Pkg
Pkg.add("CSV")

   Resolving package versions...
  No Changes to `~/CODE/pcaDCA.jl/Project.toml`
  No Changes to `~/CODE/pcaDCA.jl/Manifest.toml`


In [7]:
using CSV, DataFrames

In [8]:
df=CSV.read("../../../Downloads/dataset_filtered_paper.csv", DataFrame)


279565×21 DataFrame
    Row │ Unnamed: 0  Entry       Cross-reference (pfam)    Cross-reference (i ⋯
        │ Int64       String15    Union{Missing, String}    String             ⋯
────────┼───────────────────────────────────────────────────────────────────────
      1 │          0  O94625      PF00226;PF01556;PF00684;  IPR002939;IPR00162 ⋯
      2 │          1  P0DOJ7      PF02380;                  IPR001623;IPR03686
      3 │          2  Q9CQV7      PF00226;                  IPR001623;IPR03686
      4 │          3  Q9VTJ8      missing                   IPR001623;IPR03686
      5 │          4  O74746      PF00226;                  IPR001623;IPR03686 ⋯
      6 │          5  Q27237      PF00226;PF01556;PF00684;  IPR012724;IPR00293
      7 │          6  Q9NZJ4      PF05168;PF00240;          IPR001623;IPR03689
      8 │          7  Q9JLC8      PF05168;                  IPR001623;IPR03689
   ⋮    │     ⋮           ⋮                  ⋮                              ⋮  ⋱
 279559 │     297391  A0A7L0MF90  PF00226;PF01556;          IPR002939;IPR00162 ⋯
 279560 │     297392  A0A093EW57  PF00226;                  IPR001623;IPR04285
 279561 │     297393  A0A4W4GQH7  PF00226;PF01556;          IPR002939;IPR00162
 279562 │     297394  A0A4W4GU30  PF00226;                  IPR001623;IPR04463
 279563 │     297395  A0A4W4ERJ8  PF00226;PF05207;          IPR001623;IPR04297 ⋯
 279564 │     297396  A0A7L3P0G6  PF00226;                  IPR001623;IPR01825
 279565 │     297397  A0A8T2PQ19  PF00226;PF14901;          IPR001623;IPR03686
                                              18 columns and 279550 rows omitted

In [9]:
msa = df[!,:JD_sequence]
class = df[!, :labelABC]
loc = df[!, :Localization]
tax = df[!,"Taxonomic lineage (ALL)"]

279565-element SentinelArrays.ChainedVector{String, Vector{String}}:
 "cellular organisms, Eukaryota, " ⋯ 235 bytes ⋯ " / ATCC 24843) (Fission yeast)"
 "Viruses, Monodnaviria (single-s" ⋯ 162 bytes ⋯ "olyomavirus (strain BG) (MPyV)"
 "cellular organisms, Eukaryota, " ⋯ 368 bytes ⋯ "Mus, Mus, Mus musculus (Mouse)"
 "cellular organisms, Eukaryota, " ⋯ 452 bytes ⋯ "phila melanogaster (Fruit fly)"
 "cellular organisms, Eukaryota, " ⋯ 235 bytes ⋯ " / ATCC 24843) (Fission yeast)"
 "cellular organisms, Eukaryota, " ⋯ 452 bytes ⋯ "phila melanogaster (Fruit fly)"
 "cellular organisms, Eukaryota, " ⋯ 369 bytes ⋯ "ae, Homo, Homo sapiens (Human)"
 "cellular organisms, Eukaryota, " ⋯ 368 bytes ⋯ "Mus, Mus, Mus musculus (Mouse)"
 "Viruses, Varidnaviria, Bamfordv" ⋯ 88 bytes ⋯ "oeba polyphaga mimivirus (APMV)"
 "cellular organisms, Eukaryota, " ⋯ 139 bytes ⋯ "telium discoideum (Slime mold)"
 ⋮
 "cellular organisms, Eukaryota, " ⋯ 399 bytes ⋯ "a macroura (Pin-tailed whydah)"
 "cellular organisms, Euka

In [ ]:
map(x->split(x)[1], tax)


279565-element SentinelArrays.ChainedVector{Bool, Vector{Bool}}:
 1
 0
 1
 1
 1
 1
 1
 1
 0
 1
 ⋮
 1
 1
 1
 1
 1
 1
 1
 1
 1

In [ ]:
tax_ = []
for i in 1:length(msa)
    info = split(tax[i])[1:3]
    if info[1] == "Viruses,"
        push!(tax_, "Viruses")
    elseif info[1] == "cellular"
        if info[2] == "organisms,"
            if info[3] == "Eukaryota,"
                push!(tax_, "Eukaryota")
            elseif info[3] == "Bacteria,"
                push!(tax_, "Bacteria")
            elseif info[3] == "Archaea,"
                push!(tax_, "Archaea")
            else 
                push!(tax_, "Other")
            end  
        else 
            push!(tax_, "Other")
        end
    else
        push!(tax_, "Other")
    end

end
tax_

279565-element Vector{Any}:
 "Eukaryota"
 "Viruses"
 "Eukaryota"
 "Eukaryota"
 "Eukaryota"
 "Eukaryota"
 "Eukaryota"
 "Eukaryota"
 "Viruses"
 "Eukaryota"
 ⋮
 "Eukaryota"
 "Eukaryota"
 "Eukaryota"
 "Eukaryota"
 "Eukaryota"
 "Eukaryota"
 "Eukaryota"
 "Eukaryota"
 "Eukaryota"

In [45]:
sum(tax_ .== "Bacteria")+sum(tax_ .== "Eukaryota")+sum(tax_ .== "Viruses") + sum(tax_ .== "Archea") 

274351

In [47]:
length(msa)

279565

In [46]:
sequences = Dict(["seq$(i)" => msa[i] for i in 1:length(msa)]...)
writefasta("sequences.fasta", sequences)


UndefVarError: UndefVarError: `writefasta` not defined

In [43]:
W,Z,L,M,q = pcaDCA.read_fasta("sequences.fasta");

removing duplicate sequences... done: 279563 -> 161186
θ = 0.3470443298287371 threshold = 21.0
M = 161186 N = 63 Meff = 18514.236448124746
